# Demo GMM sobre datos Big Five (OCEAN)

Este notebook demuestra el algoritmo **Modelo de Mezcla Gaussiana (GMM)** aplicado
al dataset real recolectado a traves de la encuesta del proyecto.

Se cubre: carga, escalado, seleccion de k por BIC, entrenamiento,
probabilidades por cluster, visualizacion PCA 2D, perfil por cluster,
y deteccion de casos fronterizos.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
DATA_PATH = Path('..') / 'data' / 'real.csv'
df = pd.read_csv(DATA_PATH)
print(f'Registros: {len(df)}')
df.head()

In [ ]:
DIMENSIONS = ['O', 'C', 'E', 'A', 'N']
print('Shape:', df.shape)
df[DIMENSIONS].describe().round(3)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[DIMENSIONS].values)
print('X_scaled shape:', X_scaled.shape)
print('Media (deberia ser ~0):', X_scaled.mean(axis=0).round(3))
print('Std  (deberia ser ~1):', X_scaled.std(axis=0).round(3))

## Buscar el k optimo con BIC

El **Bayesian Information Criterion** penaliza modelos complejos. El k con menor BIC
suele indicar el mejor balance entre ajuste y parsimonia.

In [ ]:
ks = range(2, 9)
bics = []
aics = []
for k in ks:
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    gmm.fit(X_scaled)
    bics.append(gmm.bic(X_scaled))
    aics.append(gmm.aic(X_scaled))

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(ks), y=bics, mode='lines+markers', name='BIC'))
fig.add_trace(go.Scatter(x=list(ks), y=aics, mode='lines+markers', name='AIC'))
fig.update_layout(title='BIC / AIC por numero de componentes', xaxis_title='k', yaxis_title='Valor')
fig.show()

k_opt = list(ks)[int(np.argmin(bics))]
print(f'k sugerido (menor BIC): {k_opt}')

In [ ]:
gmm = GaussianMixture(n_components=k_opt, covariance_type='full', random_state=42)
gmm.fit(X_scaled)
labels = gmm.predict(X_scaled)
probs = gmm.predict_proba(X_scaled)
print(f'Distribucion de clusters:')
print(pd.Series(labels).value_counts().sort_index())

In [ ]:
prob_df = pd.DataFrame(probs.round(3), columns=[f'P(Cluster {i})' for i in range(k_opt)])
prob_df['cluster'] = labels
prob_df['prob_max'] = probs.max(axis=1).round(3)
prob_df.head(10)

## Visualizacion PCA 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)
var_expl = pca.explained_variance_ratio_
print(f'Varianza explicada: PC1={var_expl[0]*100:.1f}% · PC2={var_expl[1]*100:.1f}% · total={var_expl.sum()*100:.1f}%')

plot_df = pd.DataFrame({'PC1': X_2d[:, 0], 'PC2': X_2d[:, 1], 'Cluster': [f'C{c}' for c in labels]})
fig = px.scatter(plot_df, x='PC1', y='PC2', color='Cluster', opacity=0.85, title='Proyeccion PCA 2D')
fig.show()

## Perfil por cluster (media OCEAN)

In [ ]:
df_ = df.copy()
df_['cluster'] = labels
profile = df_.groupby('cluster')[DIMENSIONS].mean().round(2)
profile['n'] = df_.groupby('cluster').size()
profile

## Casos fronterizos

Personas cuya probabilidad maxima es baja pertenecen a mas de un cluster.
Son especialmente interesantes: perfiles hibridos, mixtos.

In [ ]:
frontier_mask = probs.max(axis=1) < 0.6
frontier_df = df[frontier_mask].copy()
frontier_df['cluster'] = labels[frontier_mask]
frontier_df['prob_max'] = probs[frontier_mask].max(axis=1).round(3)
for i in range(k_opt):
    frontier_df[f'P(C{i})'] = probs[frontier_mask, i].round(3)

print(f'Personas fronterizas: {len(frontier_df)} de {len(df)} ({100*len(frontier_df)/len(df):.1f}%)')
frontier_df[['edad', 'genero', 'arquetipo', 'cluster', 'prob_max'] + [f'P(C{i})' for i in range(k_opt)]].head(10)